In [1]:
from dotenv import load_dotenv
load_dotenv()

#loads the OpenAI client
from openai import OpenAI
openai_client = OpenAI()

In [2]:
#next, load the data and build the index
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
## Create the RAG Agent
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)
#index first, then the LLM client second

In [ ]:
#Try a question
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download  \n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and run:\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo test that the local server is running, you can also use:\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [ ]:
#Try it with a typo - we use lexical search so it looks for exact word and finds nothing. We need something smarter - an agent.
assistant.rag("How do I run Olama locally?")

'I don’t see any context about running **Olama locally**.\n\nThe closest related context is about using local/open-source models, but it doesn’t include Olama/Ollama setup steps.'

In [ ]:
#Asking the LLM without any tools. Model answers from its general knowledge. It doesn't know about the FAQ.
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Absolutely — you can usually join a course if it’s still open for enrollment.\n\nIf you want, I can help you figure out the next step. Please send me any of these:\n- the course name,\n- the platform or school,\n- the enrollment deadline,\n- or a link/screenshot of the course page.\n\nThen I can tell you whether you can still join and what to do next.'

## Defining the tool
* we define a top-level search function that queries the index directly
* we're telling the LLM - hey there's actually a "search" function you can use. This function uses the index we built earlier.

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
#Next, need to let LLM know how to use it. LLMs are language agnostic so it doesn't see the Python code, only a schema describing what the function does and what arguments it takes.
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}
#The "description" field is the most important because the model reads it to decide when to call the function

In [10]:
#Send the same question but this time with the tool in the request.
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered course join anytime enrollment late registration"}', call_id='call_2LtuchLX8rulnibKD5nB8s5U', name='search', type='function_call', id='fc_02ca6d38095a8f27006a299dd1928c8199bce255568cae9c47', namespace=None, status='completed')]

In [ ]:
#Here, the "call" function contains JSON arguments. We will parse them, call our search function, and serialize the result
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [ ]:
#send the result back to the model. Add the model's output to convo history. Then we add the tool result.
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})
#the call_id links the tool result to the specific function call the model requested.
#If the model makes multiple function calls in one turn, each one gets its own call_id

## Ask the model again
* This time, the model has the original question, its own decision to call the "search" function", and the FAQ results. 
* We send the whole history because LLMs are stateless between API calls. 
    * The memory is the list you send as input
    * If you send only the tool result, the model has no idea what's going on.
    * So on this second call, we replay everythinig we have so far.

* This is the full function-calling loop for a single turn
    * With plain RAG, we made one call. Here, we make two. 
    * Turning RAG agentic means more round-trips

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.'

## Checking the Token usage and cost

In [14]:
#the response has a usage field with the token counts:
usage = response.usage
usage.input_tokens, usage.output_tokens

(654, 33)

In [15]:
#For each model, the provider publishes a price per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


#### This usage up there is only for the second API call. The first call has its own usage and its own cost. We pay more on the second call because we resent the full history as input.
* With a real agent loop, the model can make many calls, so the costs add up. Hence, it's important to keep an eye on usage.

## Now, make multiple calls

In [ ]:
#send one more API request (loop here) <-- keep sending requests to LLM until at some point, LLM says I'm done, and ready to give an answer

#### A developer prompt. 
* Includes the words "Make multiple searches"
* Also gives a role to the agent - "You're a course teaching assistant"

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
#We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. 
#It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. 
#We only have one tool for now, so we dispatch on the function name directly.
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

### The full agent loop

In [ ]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

## Wrap it all in a function

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
#Try it with a question that has a typo:
agent_loop(instructions, "How do I run Olama locally?")

In [ ]:
#Try the course enrollment question again.
agent_loop(instructions, "I just discovered the course. Can I still join it?")